# **Loading additional datasets**: To augment analysis

_Importing libraries..._

In [1]:
# Data science libraries
import pandas as pd
import numpy as np

# Importing Hugging Face datsets
from datasets import load_dataset, concatenate_datasets

# Ability to clear console
from IPython.display import clear_output

# Importing custom pre-process text
import sys
import os

# Add the 'your_project' root directory to sys.path
# This assumes your_project is the parent of both 'data' and 'utils'
# os.path.abspath(__file__) would give you the notebook's path,
# then two '..' go up to 'your_project'
module_path = os.path.abspath(os.path.join('..', os.path.dirname('')))
if module_path not in sys.path:
    sys.path.append(module_path)

# Importing custom pre-process text
from utils.nlp import preprocess_for_hate_speech, detect_language


# **1**: Loading davidson

In [2]:
# Loading the davidson data
fname_davidson = 'labeled_data.csv'
df_davidson   = pd.read_csv(fname_davidson)

# Specifying columns we need to col
col_keep_davidson = ['hate_speech', 'tweet']
df_davidson       = df_davidson[col_keep_davidson]

# Performing a rename of the datasets
df_davidson.rename(
    columns = {
        'hate_speech' : 'label',
        'tweet'       : 'text'
    },
    inplace = True
)
# Converting label into a binary attribute
df_davidson['label'] = df_davidson['label'].apply(lambda x : 0 if x == 0 else 1)

# Specifying this is the dataset
df_davidson['dataset'] = 'Davidson'

# Checking the dataset is correct
df_davidson.head(3)

,label,text,dataset
0,0,!!! RT @mayasolovely: As a woman you shouldn't...,Davidson
1,0,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,Davidson
2,0,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,Davidson


## **2**: Tweet-eval

In [3]:
# **1**: Loading TweetEval
dataset = load_dataset("tweet_eval", "hate", split="train[:10000]")
df_tweeteval = pd.DataFrame(dataset)

# Retain required columns
df_tweeteval = df_tweeteval[["text", "label"]]
df_tweeteval = df_tweeteval.dropna()
df_tweeteval = df_tweeteval[df_tweeteval["text"].str.len() > 10]

# Apply preprocessing
df_tweeteval['processed_text'] = df_tweeteval['text'].apply(lambda x: preprocess_for_hate_speech(text=x))
df_tweeteval['language'] = df_tweeteval['processed_text'].apply(lambda x: detect_language(text=x))
df_tweeteval = df_tweeteval[df_tweeteval["language"] == "English"]

# Remap and binarize labels
df_tweeteval["label_text"] = df_tweeteval["label"].apply(lambda x: "safe" if x == 0 else "hate")
df_tweeteval["label_id"] = df_tweeteval["label_text"].map({"safe": 0, "hate": 1})
df_tweeteval = df_tweeteval.dropna()

# Add dataset name
df_tweeteval["dataset"] = "TweetEval"

# Speicfying the relevant columns
col_keep = ['label', 'text', 'dataset']
df_tweeteval = df_tweeteval[col_keep]
df_tweeteval.head()


README.md: 0.00B [00:00, ?B/s]

c:\Users\tmccl\anaconda3\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tmccl\.cache\huggingface\hub\datasets--tweet_eval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/816k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/278k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/103k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2970 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

,label,text,dataset
0,0,@user nice new signage. Are you not concerned ...,TweetEval
1,1,A woman who you fucked multiple times saying y...,TweetEval
2,1,@user @user real talk do you have eyes or were...,TweetEval
3,1,your girlfriend lookin at me like a groupie in...,TweetEval
4,0,Hysterical woman like @user,TweetEval


## **Joining datasets**: Bringing all the datasets together

In [4]:
# List of dataframes to concatenate
datasets = [df_davidson, df_tweeteval]

# Concatenating the datasets
df = pd.concat(datasets)

# Dropping null observations
df = df.dropna()

# Introduced the new processed tweets
df['processed_text'] = df['text'].apply(lambda x : preprocess_for_hate_speech(text = x))

# Checking what language
# Here, we are using full name node the code
df['language'] = df['text'].apply(lambda x : detect_language(text = x))

_Checking balance of dataset..._

In [5]:
# Checking label
df['label'].value_counts()

label
0    24846
1     8700
Name: count, dtype: int64

_Checking the language breakdown...._

In [6]:
# Investigating language
df['language'].value_counts()

language
English                    31812
Afrikaans                    292
Welsh                        142
Dutch                        122
Tagalog                      115
German                       113
Somali                       109
Indonesian                   102
Norwegian                     87
Italian                       77
French                        73
Swedish                       71
Estonian                      44
Danish                        44
Finnish                       39
Unknown                       38
Catalan                       32
Spanish                       28
Polish                        28
Turkish                       23
Portuguese                    21
Albanian                      18
Slovenian                     17
Romanian                      17
Slovak                        17
Czech                         16
Vietnamese                    15
Croatian                      12
Swahili (macrolanguage)       11
Hungarian                      6
L

_Saving the dataset..._

In [7]:
# Saving the dataset
df.to_csv('Additional_Data.csv')